# 00C Demo Setup — FabricOps I/O

Run this once in **Engineering Development** after 0B. It demonstrates the public FabricOps I/O helpers and prepares the managed source tables required by `02_pipeline`.


## 1. Load the shared Fabric configuration

The setup notebook uses the same workspace-local `00_env_config` as the rest of the Guided Demo.


In [ ]:
%run 00_env_config


## 2. Import the FabricOps I/O helpers


In [ ]:
from fabricops_kit import (
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_json,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)


## 3. Read the same "Orders" dataset via CSV, JSON, Parquet, and Excel.

All four files contain the same 120 logical Orders rows. Paths are relative to the configured Lakehouse `Files` area, so `Demo/orders.csv` resolves under `Files/Demo/orders.csv`.


In [ ]:
orders_csv_df = read_lakehouse_csv(
    "Demo/orders.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

#display(orders_csv_df)


In [ ]:
orders_json_df = read_lakehouse_json(
    "Demo/orders.json",
    store="Bronze",
    spark_session=spark,
)
#display(orders_json_df)


In [ ]:
orders_parquet_df = read_lakehouse_parquet(
    "Demo/orders.parquet",
    store="Bronze",
    spark_session=spark,
)

#display(orders_parquet_df)


In [ ]:
orders_excel_df = read_lakehouse_excel(
    "Demo/orders.xlsx",
    store="Bronze",
    spark_session=spark,
)
#display(orders_excel_df)


## 4. Read and write the demo inputs into the lakehouse and warehouse tables

`products.csv` becomes the Lakehouse lookup table. `order_history.csv` becomes the Warehouse history table.


In [ ]:
# Write the Orders table into lakehouse 

write_lakehouse_table(
    orders_csv_df,
    table_name="orders",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

#display(orders_csv_df)

# Optional Read the table that is written in the lakehouse to see its written 
orders_table_df = read_lakehouse_table(
    table_name="orders",
    store="Bronze",
    schema="demo",
    spark_session=spark,
)

#display(orders_table_df)


In [ ]:
# Read and Write the Product table into lakehouse 

products_df = read_lakehouse_csv(
    "Demo/products.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

#display(products_df)

write_lakehouse_table(
    products_df,
    table_name="products",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

products_table_df = read_lakehouse_table(
    table_name="products",
    store="Bronze",
    schema="demo",
    spark_session=spark,
)

#display(products_table_df)


In [ ]:
order_history_df = read_lakehouse_csv(
    "Demo/order_history.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

#display(order_history_df)

write_warehouse_table(
    order_history_df,
    schema="demo",
    table_name="order_history",
    store="Gold",
    mode="overwrite",
)

# read_warehouse_table() reads the full table. 
order_history_table_df = read_warehouse_table(
    schema="demo",
    table_name="order_history",
    store="Gold",
    spark_session=spark,
)

#display(order_history_table_df)

# read_warehouse_query() pushes SQL to the Warehouse before returning the result to Spark. 
order_history_sample_df = read_warehouse_query(
    "SELECT TOP 5 * FROM demo.order_history ORDER BY 1",
    store="Gold",
    spark_session=spark,
)

#display(order_history_sample_df)
